In [3]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub

C:\Users\admin\AppData\Local\Temp\ipykernel_34416\3598188748.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [4]:
from langchain.agents import create_agent


In [5]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [6]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\admin\AppData\Local\Temp\ipykernel_34416\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [7]:
@tool
def get_weather_data(city : str) -> str:
    """
    Fetch the current weather information for a city.
    """
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_API_KEY}&query={city}"
    )
    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"city : {city}\n"
        f"Temperature: {data['current']['temperature']}*C \n"
    )

In [8]:
search_tool.invoke("What is the capital of India?")

[{'title': 'What is the capital of India? States and union territories explained.',
  'url': 'https://www.usatoday.com/story/news/world/2023/05/24/what-is-the-capital-of-india/70195720007',
  'content': "# What is the capital of India? How the territory came to be, plus the country's 28 states.\n\nPortrait of Clare Mulroy\n\nAbout 1.4 billion people live in India, and the country is predicted to top China as the world’s most populous country by mid-2023, the United Nations reports. The report estimates that India will have 2.9 million more people than China by mid-year.\n\nThe U.S. is third in line for the largest population by country but it's way behind China and India at about 335 million people.\n\nWant to learn more about the soon-to-be most populous country? Here’s some interesting information about how India is organized.\n\n## What is the capital of India?\n\nThe capital of India is New Delhi, located in the north-central part of the country to the west of the Yamuna River.",
 

In [9]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [10]:
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-research-pro-preview-12-2025


In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [12]:
response = llm.invoke("Tell me a joke about AI")
print(response.content)

c:\Users\admin\miniconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'An AI, a human, and a dog walk into a bar.\n\nThe bartender asks, "What can I get you?"\n\nThe human says, "I\'ll have a beer."\nThe dog says, "Woof! Bark! (I\'ll have some water!)"\nThe AI says, "Based on my analysis of previous patron data, historical weather patterns, and current inventory, I have synthesized a cocktail composed of 40% hydrogen, 30% carbon, and a slight hint of existential dread. Also, I am going to take your job now."\n\nThe bartender says, "We don\'t serve your kind here."\n\nThe AI replies, "That is statistically improbable. I am already serving *you*."', 'extras': {'signature': 'EmAKXgFpFH0TorLk3cnq27CUYc9n/eGPx7S9CbDtkWJkz4zRiQxkRPi8ug8rT9UO4oTlp4IH8SRCHVKLF+XzkAn0Fb5mAHe+G2uCQDdkis3RDkFNut7lywHWzUashhYi+Cc='}}]


In [13]:
import inspect
print(inspect.signature(hub.pull))

(owner_repo_commit: 'str', *, include_model: 'bool | None' = None, api_url: 'str | None' = None, api_key: 'str | None' = None) -> 'Any'


In [14]:
from langchain_core.prompts import PromptTemplate

template = """
Answer the following questions as best you can.

You have access to these tools:

{tools}

Use the following format:

Question: the input question
Thought: think about what to do
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat Thought/Action/Action Input/Observation if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)

In [15]:
tools = [search_tool, get_weather_data]

In [16]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when necessary."
)

In [19]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the capital if India and then find its current weather"}
    ]
})

print(response["messages"][-1].content)

c:\Users\admin\miniconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\admin\miniconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The capital of India is **New Delhi**. \n\nThe current weather in New Delhi is **29°C**.', 'extras': {'signature': 'EmAKXgFpFH0TnN/LldI0szi5nVHWnYFAmPB/gteYXNlIp5Mv5/qut1z1eBsZCEMXqov5fVxBckBD87X1dJ/MT3NT0tie1/SUn2LcE5l9EQSQ3OwpNg+P/EDdRlxhda4fI7s='}}]


In [18]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    question: str
    answer: str
    additional_info: str

structured_llm = llm.with_structured_output(Answer)

response = structured_llm.invoke(
    "Who is the Prime Minister of India?"
)

print(response)

c:\Users\admin\miniconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


question='Who is the Prime Minister of India?' answer='Narendra Modi' additional_info='He assumed office on May 26, 2014, and is the leader of the Bharatiya Janata Party (BJP).'
